# Проверка X_INN по трем выгрузкам

Тетрадка загружает 3 файла, применяет фильтрацию по методике `analysis_crm_segments.ipynb`, считает уникальные `X_INN` и проверяет пересечения множеств ИНН.

In [ ]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

In [ ]:
# Пути к данным (по вашей договоренности)
DATA_DIR = Path("/home/jovyan/documents/DMUKP_FP_DEF/data")

FILES = {
    "crm_last_version_inn": DATA_DIR / "crm_last_version_inn.csv",
    "crm_all_clients": DATA_DIR / "Выгрузка_CRM_все_клиенты.csv",
    "fp_by_inn": DATA_DIR / "Выгрузка_ФП_по_ИНН.xlsx",
}

for name, path in FILES.items():
    print(f"{name}: {path}")

In [ ]:
DATE_FROM = pd.Timestamp("2023-01-01")
DATE_TO = pd.Timestamp("2025-12-31")
ALLOWED_SOURCES = ["Н2.0", "Справочник CRM-системы", "CRM-система"]

SEGMENT_MAP = {
    "ДМСБ (микро)": "МкБ",
    "ДМСБ (малый)": "МСБ",
    "ДМСБ (средний)": "МСБ",
    "ДКБ": "ККБ",
}


def normalize_inn(s):
    if pd.isna(s):
        return None
    s = str(s).strip()
    if s.endswith(".0"):
        s = s[:-2]
    s = s.lstrip("0") or "0"
    if len(s) <= 10:
        return s.zfill(10)
    return s.zfill(12)


def prepare_df(raw_df: pd.DataFrame, file_label: str) -> tuple[pd.DataFrame, dict]:
    """Подготовка данных с разной логикой для разных источников."""
    df = raw_df.copy()
    stats = {"file": file_label, "rows_raw": len(df)}

    if "X_INN" not in df.columns:
        raise KeyError(f"В файле {file_label} нет колонки X_INN")

    # Базовая подготовка для всех файлов
    df["inn"] = df["X_INN"].apply(normalize_inn)

    if "X_AREA_RESP" in df.columns:
        df["X_AREA_RESP"] = df["X_AREA_RESP"].astype(str).str.strip()
        df["segment"] = df["X_AREA_RESP"].map(SEGMENT_MAP)

    # Для all_clients используем логику analysis_crm_all_clients:
    # только нормализация и дедуп по inn, без фильтров segments (дата/VAL/segment notna/FP-поля).
    if file_label == "crm_all_clients":
        df = df.dropna(subset=["inn"]).copy()
        df = df.drop_duplicates(subset=["inn"]).copy()

    else:
        # Логика analysis_crm_segments для CRM/FP источников
        if "IDENTIFICATION_DATE" in df.columns:
            df["dt"] = pd.to_datetime(df["IDENTIFICATION_DATE"], dayfirst=True, errors="coerce")
            df = df[(df["dt"] >= DATE_FROM) & (df["dt"] <= DATE_TO)].copy()

        if "VAL" in df.columns:
            df = df[df["VAL"].astype(str).str.strip().isin(ALLOWED_SOURCES)].copy()

        if "TYPE_FP" in df.columns:
            df["TYPE_FP"] = df["TYPE_FP"].astype(str).str.strip().replace({"FP": "ФП", "SFP": "СФП"})

        if "X_AREA_RESP" in df.columns:
            df = df[df["segment"].notna()].copy()

        if "NUMBER_FP_SFP" in df.columns:
            df["fp_num"] = df["NUMBER_FP_SFP"].astype(str).str.strip()
            df = df.dropna(subset=["inn", "NUMBER_FP_SFP"]).copy()

        dedup_cols = [c for c in ["inn", "fp_num", "TYPE_FP", "IDENTIFICATION_DATE"] if c in df.columns]
        if dedup_cols:
            df = df.drop_duplicates(subset=dedup_cols).copy()
        else:
            df = df.drop_duplicates(subset=["inn"]).copy()

    stats["rows_filtered"] = len(df)
    stats["unique_x_inn"] = df["X_INN"].astype(str).nunique(dropna=True)
    stats["unique_inn_norm"] = df["inn"].nunique(dropna=True)

    return df, stats

In [ ]:
def read_semicolon_keep_all(path: Path, skip_rows: int = 33, sep: str = ";") -> pd.DataFrame:
    """Читает CSV после skip_rows без потери строк (по аналогии с analysis_crm_all_clients)."""
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        lines = f.readlines()

    if len(lines) <= skip_rows:
        raise ValueError(f"Недостаточно строк после skip_rows={skip_rows}: {path}")

    header = lines[skip_rows].rstrip("\n")
    columns = header.split(sep)
    ncols = len(columns)

    comment_candidates = ["comment", "COMMENT", "COMMENT_TEXT"]
    comment_idx = next((columns.index(c) for c in comment_candidates if c in columns), None)

    records = []
    fixed_more = 0
    fixed_less = 0

    for raw_line in lines[skip_rows + 1 :]:
        raw = raw_line.rstrip("\n")
        parts = raw.split(sep)

        if len(parts) == ncols:
            records.append(parts)
            continue

        if len(parts) > ncols and comment_idx is not None:
            extra = len(parts) - ncols
            merged_comment = sep.join(parts[comment_idx : comment_idx + extra + 1])
            repaired = parts[:comment_idx] + [merged_comment] + parts[comment_idx + extra + 1 :]
            if len(repaired) == ncols:
                records.append(repaired)
                fixed_more += 1
                continue

        if len(parts) < ncols:
            records.append(parts + [""] * (ncols - len(parts)))
            fixed_less += 1
            continue

        tail = sep.join(parts[ncols - 1 :])
        repaired = parts[: ncols - 1] + [tail]
        records.append(repaired)
        fixed_more += 1

    df_out = pd.DataFrame(records, columns=columns)
    print(
        f"{path.name}: rows={len(df_out):,}, cols={df_out.shape[1]}, "
        f"fixed_more={fixed_more:,}, fixed_less={fixed_less:,}"
    )
    return df_out


def load_file(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Файл не найден: {path}")

    if path.suffix.lower() == ".csv":
        # Для all_clients используем спец-чтение из analysis_crm_all_clients.ipynb
        if path.name == "Выгрузка_CRM_все_клиенты.csv":
            return read_semicolon_keep_all(path=path, skip_rows=33, sep=";")

        # Базовый вариант из методики segments
        return pd.read_csv(path, dtype=str, low_memory=False)

    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path, dtype=str)

    raise ValueError(f"Неподдерживаемый формат: {path}")


raw_data = {}
prepared_data = {}
stats_rows = []

for file_key, file_path in FILES.items():
    raw_df = load_file(file_path)
    df_prepared, stats = prepare_df(raw_df, file_key)

    raw_data[file_key] = raw_df
    prepared_data[file_key] = df_prepared
    stats_rows.append(stats)

    print(f"{file_key}: raw={stats['rows_raw']:,}, filtered={stats['rows_filtered']:,}, unique_inn={stats['unique_inn_norm']:,}")

In [ ]:
stats_df = pd.DataFrame(stats_rows)
stats_df = stats_df[["file", "rows_raw", "rows_filtered", "unique_x_inn", "unique_inn_norm"]]
stats_df = stats_df.sort_values("file").reset_index(drop=True)

print("Статистика по файлам:")
display(stats_df)

In [ ]:
sets = {k: set(v["inn"].dropna().unique()) for k, v in prepared_data.items()}

A = sets["crm_last_version_inn"]
B = sets["crm_all_clients"]
C = sets["fp_by_inn"]

overlap_stats = pd.DataFrame([
    {"metric": "|A| crm_last_version_inn", "value": len(A)},
    {"metric": "|B| crm_all_clients", "value": len(B)},
    {"metric": "|C| fp_by_inn", "value": len(C)},
    {"metric": "|A ∩ B|", "value": len(A & B)},
    {"metric": "|A ∩ C|", "value": len(A & C)},
    {"metric": "|B ∩ C|", "value": len(B & C)},
    {"metric": "|A ∩ B ∩ C|", "value": len(A & B & C)},
    {"metric": "|A \\ B|", "value": len(A - B)},
    {"metric": "|A \\ C|", "value": len(A - C)},
    {"metric": "|B \\ A|", "value": len(B - A)},
    {"metric": "|B \\ C|", "value": len(B - C)},
    {"metric": "|C \\ A|", "value": len(C - A)},
    {"metric": "|C \\ B|", "value": len(C - B)},
])

expected_c = A - B
check_expected = pd.DataFrame([
    {"metric": "|expected_c = A \\ B|", "value": len(expected_c)},
    {"metric": "|C|", "value": len(C)},
    {"metric": "C == expected_c", "value": C == expected_c},
    {"metric": "|C \\ expected_c|", "value": len(C - expected_c)},
    {"metric": "|expected_c \\ C|", "value": len(expected_c - C)},
])

print("Пересечения и разности:")
display(overlap_stats)
print("Проверка, что fp_by_inn == crm_last_version_inn - crm_all_clients:")
display(check_expected)

In [ ]:
# Показываем примеры расхождений (если есть)
examples = {
    "C_minus_expected": sorted(list(C - expected_c))[:50],
    "expected_minus_C": sorted(list(expected_c - C))[:50],
    "A_minus_B": sorted(list(A - B))[:50],
}

for key, values in examples.items():
    print(f"\n{key}: {len(values)} примеров")
    print(values[:10])

samples_df = pd.DataFrame({
    "C_minus_expected": pd.Series(examples["C_minus_expected"]),
    "expected_minus_C": pd.Series(examples["expected_minus_C"]),
    "A_minus_B": pd.Series(examples["A_minus_B"]),
})

display(samples_df.head(50))

In [ ]:
# Опциональный экспорт результатов
EXPORT_RESULTS = False
EXPORT_PATH = Path("e:/DMUKP/final_report/crm_inn_crosscheck_results.xlsx")

if EXPORT_RESULTS:
    with pd.ExcelWriter(EXPORT_PATH, engine="openpyxl") as writer:
        stats_df.to_excel(writer, sheet_name="inn_stats", index=False)
        overlap_stats.to_excel(writer, sheet_name="set_overlaps", index=False)
        check_expected.to_excel(writer, sheet_name="expected_check", index=False)
        samples_df.to_excel(writer, sheet_name="sample_diffs", index=False)
    print(f"Результаты сохранены: {EXPORT_PATH}")
else:
    print("Экспорт выключен. Установите EXPORT_RESULTS = True, чтобы сохранить файл.")